# Investigating RealtimeSTT

This notebook explores the [RealtimeSTT](https://github.com/KoljaB/RealtimeSTT) library to test if it can provide lower latency transcription for the live buzzword counter.

RealtimeSTT is built on top of `faster-whisper` (by default) and includes built-in Voice Activity Detection (VAD) and realtime text updates, making it a great candidate for our live odometer.

In [ ]:
# 1. Install Dependencies
# Note: macOS users might need 'brew install portaudio' if PyAudio isn't installed.
!pip install "RealtimeSTT[faster-whisper]"

In [ ]:
import ssl
# Fix for macOS SSL certificate issues when downloading models from torch hub
ssl._create_default_https_context = ssl._create_unverified_context

from RealtimeSTT import AudioToTextRecorder
import time
from IPython.display import clear_output

### Basic Usage with Realtime Callbacks
Here we define a simple callback function that prints partial transcripts as you speak, allowing for "instant" feedback before the final refined sentence is returned.

In [ ]:
def process_realtime_text(text):
    # This fires repeatedly as you are speaking (partial transcripts)
    print(f"\r[Live] {text}", end="", flush=True)

print("Initializing RealtimeSTT Recorder (loading model)...")
recorder = AudioToTextRecorder(
    model="small",               # Whisper model size
    language="en",               # Force english for testing
    enable_realtime_transcription=True,
    on_realtime_transcription_update=process_realtime_text,
    realtime_model_type="tiny.en" # Use a tiny model for live updates, 'small' for final
)
print("Ready!")

In [ ]:
print("\n--- Speak into your microphone! (Stop cell to exit) ---")

try:
    while True:
        # The text() method blocks until a full sentence/turn is completed and VAD detects a pause.
        # While it blocks, the process_realtime_text callback will fire!
        final_text = recorder.text()
        print(f"\n\n=> [Final] {final_text}\n")
except KeyboardInterrupt:
    print("\n\nStopped listening.")
finally:
    recorder.shutdown()

## Conclusion & Integration Notes

If this latency feels snappier than our custom `speech_recognition` + manual chunking approach, we can easily swap it into `app.py`!

**Why it might be faster:**
RealtimeSTT uses a dual-model approach. It runs an extremely fast model (like `tiny`) constantly to fire off the `on_realtime_transcription_update` callbacks, giving you instant visual feedback. Then, when it detects you've paused speaking, it runs the higher quality model (like `small` or `medium`) over the entire chunk to give you the finalized transcript.